# Tổng hợp dự đoán và thử kết hợp mô hình

**Phụ trách** Mai tổng hợp dự đoán, chọn trọng số bằng OOF và xuất submission ensemble. Dự đoán đầu vào đến từ Gradient Boosting của Mai, SVR của Nam, Random Forest của Như và ElasticNet của Thái.

So sánh dự đoán của Gradient Boosting, SVR, Random Forest và ElasticNet trên cùng 5 fold. Trọng số được chọn bằng dự đoán ngoài fold (OOF), sau đó mới đánh giá trên holdout và xuất dự đoán test.

## 1. Kiểm tra các file dự đoán

Bốn file submission được đối chiếu với `sample_submission.csv` theo `Id`. Mỗi thành viên cần có đúng một dự đoán giá dương cho mỗi căn trong tập test. Thứ tự dòng có thể khác nhau giữa các file, vì bước kết hợp sẽ căn chỉnh theo `Id`.

In [ ]:
from pathlib import Path
import subprocess
import sys

import numpy as np
import pandas as pd
from IPython.display import display

cwd = Path.cwd()
root = next(
    (folder for folder in (cwd, *cwd.parents)
     if (folder / "members/Mai/src/run_ensemble.py").is_file()),
    None,
)
if root is None:
    raise FileNotFoundError("Hãy mở notebook từ thư mục Challenge1 hoặc một thư mục con.")

sample = pd.read_csv(root / "data/raw/sample_submission.csv")
submissions_dir = root / "results/submissions"
files = {
    "Gradient Boosting": "submission_Mai.csv",
    "SVR": "submission_Nam_SVR.csv",
    "Random Forest": "submission_Nhu.csv",
    "ElasticNet": "submission_Thai.csv",
}

checks = []
for model, filename in files.items():
    predictions = pd.read_csv(submissions_dir / filename)
    ids_match = predictions["Id"].is_unique and set(predictions["Id"]) == set(sample["Id"])
    prices_valid = np.isfinite(predictions["SalePrice"]).all() and predictions["SalePrice"].gt(0).all()
    if not ids_match or not prices_valid:
        raise ValueError(f"File dự đoán không hợp lệ: {filename}")
    checks.append({"Mô hình": model, "File": filename, "Số dự đoán": len(predictions)})

display(pd.DataFrame(checks))

## 2. Chọn trọng số bằng OOF và đánh giá holdout

Script `run_ensemble.py` ghép OOF của 1.168 căn development theo `Id`, đối chiếu nhãn và kiểm tra điểm từng mô hình trên 5 fold chung. Các dự đoán được kết hợp trên thang log.

Bảng so sánh gồm trung bình đều và các bộ trọng số không âm có tổng bằng 1, thử theo bước 0,10. Cấu hình có RMSE log CV thấp nhất được chọn trước khi tính RMSE log và MAE trên 292 căn holdout. Ô này cũng cập nhật bảng điểm và file submission ensemble.

In [ ]:
run = subprocess.run(
    [sys.executable, "-B", str(root / "members/Mai/src/run_ensemble.py")],
    cwd=root, capture_output=True, encoding="utf-8", errors="replace",
)
print(run.stdout, end="")
if run.returncode != 0:
    print(run.stderr, end="")
    run.check_returncode()

## 3. So sánh các phương án

Các dòng có trọng số 100% ở một mô hình là điểm của mô hình riêng. Bảng dưới so sánh mô hình riêng tốt nhất với trung bình đều và phương án được chọn. Bảng thứ hai hiển thị các bộ trọng số có CV thấp nhất.

In [ ]:
scores = pd.read_csv(root / "ensemble/oof_comparison.csv")
weight_columns = ["Mai", "Nam", "Nhu", "Thai"]
single_models = scores.loc[scores[weight_columns].eq(1).any(axis=1)]
equal_weights = scores.loc[scores[weight_columns].eq(0.25).all(axis=1)]
best_single = single_models.loc[single_models["cv_rmse_log"].idxmin()]
best = scores.loc[scores["cv_rmse_log"].idxmin()]

comparison = pd.DataFrame({
    "Mô hình riêng tốt nhất": best_single,
    "Trung bình đều": equal_weights.iloc[0],
    "Trọng số được chọn": best,
}).T
display(comparison.round(5))
display(scores.head(5).round(5))

## 4. Kiểm tra file nộp

File `submission_ensemble.csv` chứa giá đã chuyển về thang gốc bằng `exp()`. Kiểm tra số cột, số dòng, thứ tự `Id` và giá trước khi dùng file để nộp.

In [ ]:
output = submissions_dir / "submission_ensemble.csv"
submission = pd.read_csv(output)
assert list(submission.columns) == ["Id", "SalePrice"]
assert submission["Id"].equals(sample["Id"])
assert np.isfinite(submission["SalePrice"]).all() and submission["SalePrice"].gt(0).all()

print(f"{output.name}: {len(submission):,} dự đoán hợp lệ")
display(submission.head())

## 5. Nhận xét

Trong lần chạy đã ghi nhận, trung bình đều đạt RMSE log CV 0.12062. Phương án được chọn dùng 30% Gradient Boosting, 50% SVR và 20% ElasticNet, đạt 0.11780. Random Forest có trọng số 0 ở cấu hình này. Holdout đạt RMSE log 0.11072 và MAE 13.707 trên giá gốc.

Do OOF được dùng để chọn trọng số, điểm CV của cấu hình thắng có thể lạc quan. Kết quả holdout là phép kiểm tra sau khi chốt trọng số.